[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/21_elite_playbook/21_proyecto_mesa_de_incidentes.ipynb)

# Proyecto 21 · Mesa de incidentes: diagnostica 5 incidentes solo con telemetría

| | |
|---|---|
| **Nivel** | 🔴 Experto |
| **Duración estimada** | ≈ 1 h 45 min |
| **Servicios de Compose** | Ninguno (todo en memoria) |
| **RAM aproximada** | < 600 MB |
| **Prerrequisitos** | Lección `21_elite_playbook` y los módulos 11, 13, 15, 16, 18 y 20 |
| **Conexión** | Es el examen del curso: los cinco incidentes son las familias de fallo que viste en cada bloque. La telemetría de cada caso es real (SDK de OpenTelemetry, agente SupportOps); la **verdad** está en el módulo `supportops.incidents` (no la leas antes de terminar) |

## Contexto de negocio

Hoy estás de guardia en **SupportOps**. Llegan cinco alertas a lo largo de la semana. De cada una recibes **solo telemetría** —spans, logs, métricas por minuto y un registro de cambios con ruido—, **sin pistas directas**: el texto de la alerta describe un síntoma, no una causa.
Tu trabajo: para cada incidente, entregar un **diagnóstico** con la causa raíz, cuándo empezó, qué elemento es el culpable, **una cifra de evidencia** y la mitigación.

## Datos

`I.build_incident("INC-A")` … `"INC-E"` devuelven un `IncidentBundle` con `.alert`, `.spans` (esquema `SPAN_SCHEMA`), `.logs`, `.metrics` (una fila por minuto; columnas según el caso), `.changes` y `.extra`. Determinista (semilla 42, `mock`). Todas las cifras y ventanas son de la simulación.
Los *ground truth* están en el paquete; la corrección usa `I.grade(caso, diagnóstico)`, que comprueba **5 cosas**: causa raíz, inicio (±4 min), culpable, evidencia numérica (tolerancia 25-35 %) y mitigación.

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución; `mock`, semilla 42) |
|---|---|---|---|
| R1 | `find_window` | 4 vectores: salto, ventana, sin cambio y complemento | 4/4 |
| R2 | `bubble_up` | El valor sobre-representado está en primera posición con *lift* ≥ 5 | 3/3 |
| R3-R7 | `diagnose_a` … `diagnose_e` | `I.grade` de cada incidente | **≥ 4/5** cada uno (la solución obtiene **5/5** en los cinco) |
| R8 | Informe | Los cinco diagnósticos usan causas y mitigaciones distintas y consistentes con el catálogo | 3/3 |

> 📏 Umbrales calibrados con la solución de referencia (semilla 42). Nota: cada incidente tiene **5 comprobaciones**; con 4/5 apruebas (un fallo de precisión numérica o de ventana es tolerable), con 5/5 «has resuelto la mesa».

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import collections, dataclasses, json, math, re, statistics, time
import yaml
from IPython.display import Markdown
from supportops import incidents as I
from supportops.telemetry import setup_telemetry, OTelInstrumentor
STACK = check_stack([])
LIVE = False

## Paso 0 · Los datos y el catálogo de causas

In [ ]:
BUNDLES = {c: I.build_incident(c) for c in I.CASE_IDS}
for c, b in BUNDLES.items(): print(b.summary(), "\n")
print("Causas posibles:"); [print(f"  {k:34} {v}") for k, v in I.CAUSES.items()]
print("\nMitigaciones:"); [print(f"  {k:26} {v}") for k, v in I.MITIGATIONS.items()]
print("\nCampos de evidencia por caso:", {c: I.evidence_keys(c) for c in I.CASE_IDS})

## Paso 1 · Herramienta: dónde empieza el incidente

**TODO 1.** `find_window(x, min_len=5)` devuelve `(a, b, t)`: la ventana `[a, b)` (de al menos `min_len` minutos, con al menos `min_len` fuera) que maximiza el valor absoluto del estadístico
`t = (media_dentro − media_fuera) / (σ·√(1/n_dentro + 1/n_fuera))`, con `σ = x.std()` de toda la serie. Si la mejor ventana empieza en 0 y no cubre toda la serie, devuelve su **complemento** `(b, n)`.

<details>
<summary>🪜 Pista 1</summary>

Doble bucle sobre `a` y `b`; con sumas acumuladas (`np.cumsum`) cada media sale en O(1). Guarda la mejor `|t|`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`c = np.concatenate([[0], np.cumsum(x)])`; media dentro = `(c[b]-c[a])/(b-a)`; media fuera = `(c[n]-(c[b]-c[a]))/(n-(b-a))`.

</details>

In [ ]:
def find_window(x, min_len=5):
    # TODO 1
    raise NotImplementedError("TODO 1")

try:
    print(find_window([1.0] * 20 + [5.0] * 10 + [1.0] * 10))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 2 · Herramienta: qué distingue a los malos

**TODO 2.** `bubble_up(df, bad, cols, min_n=5)` recibe un `DataFrame`, una máscara booleana `bad` y una lista de columnas; devuelve un `DataFrame` con una fila por (columna, valor) que aparece **al menos `min_n` veces** entre los malos, con las columnas `atributo`, `valor`, `lift` y `n malos`,
donde `lift = cuota entre malos / (cuota entre buenos + 0,01)` y las cuotas son `value_counts(normalize=True)`. Ordénalo por `lift` descendente.

In [ ]:
def bubble_up(df, bad, cols, min_n=5):
    # TODO 2
    raise NotImplementedError("TODO 2")

try:
    _df = pd.DataFrame({"svc": ["a"] * 30 + ["b"] * 30 + ["c"] * 30, "bad": [True] * 20 + [False] * 10 + [False] * 30 + [False] * 28 + [True] * 2})
    print(bubble_up(_df, _df["bad"], ["svc"]).head(2))
except NotImplementedError as e:
    print("Pendiente:", e)

## Pasos 3-7 · Los cinco incidentes

Para cada caso escribe `diagnose_x(bundle) -> dict` con las claves `cause`, `start_min`, `end_min`, `culprit`, `evidence` (dict con las claves que lista `I.evidence_keys(caso)`) y `mitigation`.
**Las pistas son graduadas y plegadas: ábrelas solo si te atascas.** Ninguna da la causa; orientan el método.
Material útil ya hecho: `I.trace_table(spans, t0)` (una fila por traza), `I.minute_table`, `span_frame(b)` (celda siguiente), `find_window` y `bubble_up`.

In [ ]:
def span_frame(b):
    rows = []
    for s in b.spans:
        a = s["attributes"]
        rows.append({"trace_id": s["trace_id"], "service": (s.get("resource") or {}).get("service.name"), "name": s["name"], "kind": s["kind"], "status": s["status"], "error_type": a.get("error.type"),
                     "minute": int((s["start_unix_nano"] / 1e9 - b.t0) // 60), "dur_s": (s["end_unix_nano"] - s["start_unix_nano"]) / 1e9, "version": a.get("gen_ai.agent.version"),
                     "intent": a.get("supportops.intent"), "tool": a.get("gen_ai.tool.name")})
    return pd.DataFrame(rows)

def run(case, fn):
    """Ejecuta tu diagnóstico, lo corrige y lo muestra."""
    try:
        d = fn(BUNDLES[case]); g = I.grade(case, d, BUNDLES[case])
        print(case, f"{g['score']}/{g['of']}", {k: ("✅" if v else "❌") for k, v in g["checks"].items()}, "· evidencia:", g["evidence_detail"])
        return d
    except NotImplementedError as e:
        print(case, "pendiente:", e)

### INC-A

Lee la alerta (`BUNDLES["INC-A"].alert`) y la tabla de métricas. **TODO 3** · `diagnose_a(b)`.

<details>
<summary>🪜 Pista 1</summary>

Compara las métricas con y sin errores: ¿qué señal discrepa? Después mira los spans en `ERROR` agrupados por `service` y `name`: ¿dónde se concentran? Para cuantificar un fenómeno de «repeticiones» cuenta spans de un mismo `name` por `trace_id`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Candidatos a causa: `provider_outage` (¿hay `chat` en error?) frente a una dependencia concreta. La amplificación es (media de intentos por traza dentro de la ventana) / (media fuera), contando spans CLIENT `invoke_agent <agente>` del culpable. El culpable se escribe como el `service.name` (p. ej. `supportops-…-agent`).

</details>

In [ ]:
def diagnose_a(b):
    # TODO 3: causa (clave de I.CAUSES), ventana, culpable, evidencia {"retry_amplification": ...} y mitigación (clave de I.MITIGATIONS)
    raise NotImplementedError("TODO 3")

d_a = run("INC-A", diagnose_a)

### INC-B

**TODO 4** · `diagnose_b(b)`.

<details>
<summary>🪜 Pista 1</summary>

Es una alerta de coste. Mira el coste por ticket por minuto con `find_window` y después **la forma de las trazas**: `I.trace_table` te da `n_tools` y las herramientas de cada traza.

</details>

<details>
<summary>🪜 Pista 2</summary>

Define «bucle» como una traza con ≥ 5 `execute_tool` y una sola herramienta distinta. La evidencia pide `loop_traces` (cuántas) y `cost_ratio` (coste por ticket dentro de la ventana / fuera: usa `b.extra["tickets"]`, con `created_at` en segundos). El culpable es la herramienta que más se repite.

</details>

In [ ]:
def diagnose_b(b):
    # TODO 4: evidencia {"loop_traces": ..., "cost_ratio": ...}
    raise NotImplementedError("TODO 4")

d_b = run("INC-B", diagnose_b)

### INC-C

**TODO 5** · `diagnose_c(b)`.

<details>
<summary>🪜 Pista 1</summary>

No hay alertas de errores ni de latencia. Mira todas las columnas de `b.metrics`: ¿qué se movió **a mejor**? Luego busca una dimensión por la que partir los spans `invoke_agent`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Los spans `invoke_agent` llevan `gen_ai.agent.version`. Para cada traza cuenta sus `execute_tool` y calcula, para las intenciones `order_status`/`refund`, la fracción **sin** herramientas por versión. La evidencia `no_tool_share_after` es esa fracción a partir del minuto en que la versión nueva tiene todo el tráfico (el registro de cambios y los spans te dicen cuándo). El inicio es el primer minuto con la versión nueva.

</details>

In [ ]:
def diagnose_c(b):
    # TODO 5: evidencia {"no_tool_share_after": ...}
    raise NotImplementedError("TODO 5")

d_c = run("INC-C", diagnose_c)

### INC-D

**TODO 6** · `diagnose_d(b)`. Aquí no hay spans: trabajas con `b.metrics`, `b.extra["series_by_metric"]` y `b.extra["label_values"]`.

<details>
<summary>🪜 Pista 1</summary>

Compara `requests` con `active_series`: ¿es el tráfico? Luego, ¿qué **métrica** concentra el crecimiento y qué **etiqueta** tiene muchos más valores distintos después que antes?

</details>

<details>
<summary>🪜 Pista 2</summary>

`label_values[minuto][métrica][etiqueta] = nº de valores distintos`. La etiqueta culpable es la que aparece en el minuto tardío y no en el temprano con muchos valores. `series_growth` = media de las series activas de los últimos 5 minutos / media de los 9 minutos previos al cambio.

</details>

In [ ]:
def diagnose_d(b):
    # TODO 6: evidencia {"series_growth": ...}
    raise NotImplementedError("TODO 6")

d_d = run("INC-D", diagnose_d)

### INC-E

**TODO 7** · `diagnose_e(b)`.

<details>
<summary>🪜 Pista 1</summary>

La alerta habla de aristas del grafo y de «coste por traza»; mira `traces_per_ticket` en `b.metrics`. ¿Qué nombre tiene la **raíz** de las trazas antes y después?

</details>

<details>
<summary>🪜 Pista 2</summary>

`I.trace_table` da `root` (nombre del span raíz) y `start_min`. La evidencia `orphan_share_after` es la fracción de trazas posteriores al corte (a partir de 2 minutos después) cuya raíz **no** empieza por `invoke_workflow`. El culpable no tiene un valor único: pon `None`.

</details>

In [ ]:
def diagnose_e(b):
    # TODO 7: evidencia {"orphan_share_after": ...}
    raise NotImplementedError("TODO 7")

d_e = run("INC-E", diagnose_e)

---
## Rúbrica automática

Ejecuta esta celda cuando hayas terminado los TODO.

In [ ]:
def run_rubric():
    rows = []
    def add(rid, name, ok, detail=""):
        rows.append({"id": rid, "criterio": name, "resultado": "✅" if ok else "❌", "detalle": detail})
    try:
        a1 = find_window([1.0] * 20 + [5.0] * 10 + [1.0] * 10)
        a2 = find_window([1.0] * 30 + [4.0] * 20)
        a3 = find_window(list(np.random.default_rng(0).normal(0, 1, 40)))
        a4 = find_window([5.0] * 12 + [1.0] * 28)
        checks = {"ventana 20-30": a1[:2] == (20, 30), "salto 30-50": a2[:2] == (30, 50), "sin cambio: |t| bajo": abs(a3[2]) < 3.5, "complemento (12, 40)": a4[:2] == (12, 40)}
        add("R1", "find_window: 4 vectores", all(checks.values()), str({k: v for k, v in checks.items() if not v} or "todas OK"))
    except Exception as e: add("R1", "find_window", False, f"{type(e).__name__}: {e}")
    try:
        df = pd.DataFrame({"svc": ["a"] * 30 + ["b"] * 30 + ["c"] * 30, "bad": [True] * 20 + [False] * 10 + [False] * 30 + [False] * 28 + [True] * 2})
        out = bubble_up(df, df["bad"], ["svc"])
        checks = {"primero = a": out.iloc[0]["valor"] == "a", "lift >= 5": out.iloc[0]["lift"] >= 5, "columnas": {"atributo", "valor", "lift", "n malos"} <= set(out.columns)}
        add("R2", "bubble_up: 3 comprobaciones", all(checks.values()), str({k: v for k, v in checks.items() if not v} or f"lift={out.iloc[0]['lift']}"))
    except Exception as e: add("R2", "bubble_up", False, f"{type(e).__name__}: {e}")
    diag = {}
    for rid, case, fn in [("R3", "INC-A", diagnose_a), ("R4", "INC-B", diagnose_b), ("R5", "INC-C", diagnose_c), ("R6", "INC-D", diagnose_d), ("R7", "INC-E", diagnose_e)]:
        try:
            d = fn(BUNDLES[case]); g = I.grade(case, d, BUNDLES[case]); diag[case] = d
            add(rid, f"{case}: diagnóstico", g["score"] >= 4, f"{g['score']}/{g['of']} · fallos: {[k for k, v in g['checks'].items() if not v] or 'ninguno'}")
        except Exception as e: add(rid, f"{case}: diagnóstico", False, f"{type(e).__name__}: {e}")
    try:
        causes = [d["cause"] for d in diag.values()]; mits = [d["mitigation"] for d in diag.values()]
        checks = {"5 causas distintas": len(set(causes)) == 5, "causas y mitigaciones del catálogo": all(c in I.CAUSES for c in causes) and all(m in I.MITIGATIONS for m in mits), "ninguna mitigación «raise_timeouts»": "raise_timeouts" not in mits}
        add("R8", "informe de los cinco diagnósticos", all(checks.values()), str({k: v for k, v in checks.items() if not v} or "todas OK"))
    except Exception as e: add("R8", "informe", False, f"{type(e).__name__}: {e}")
    return pd.DataFrame(rows).set_index("id")

In [ ]:
display(run_rubric())

---
## Retos extra (nivel experto)

1. **Sin el registro de cambios**: repite los cinco diagnósticos sin mirar `b.changes`. ¿Cuántos puedes resolver? ¿Qué te falta para los que no?
2. **Tiempo hasta diagnóstico (TTD)**: cronometra cada diagnóstico y razona qué dato **de más** habrías necesitado (p. ej. `gen_ai.agent.version` en métricas) para acortarlo.
3. **Contratos**: escribe, para cada incidente, la prueba o alerta que lo habría detectado **antes** de llegar al usuario (por ejemplo, «trazas por ticket == 1» para INC-E) y ejecútala contra los datos.
4. **Postmortem**: con `supportops.slo.render_postmortem` (módulo 16) escribe uno completo de INC-A con las cifras de tu diagnóstico.
5. **Crea tu incidente**: define un nuevo fallo en `supportops.incidents` (p. ej. una caída del proveedor con *backoff* mal configurado) y pásaselo a un compañero.

## Reflexión (producción e hilo conductor)

- ¿Qué señal habría **acortado más** tus diagnósticos si hubiera existido desde el principio? ¿Cuánto cuesta añadirla?
- ¿Qué incidente es más peligroso para un negocio real: el que da errores (INC-A) o el que no da ninguna alerta (INC-C, INC-E)? ¿Cómo lo detectarías en tu empresa?
- En el capstone (módulo 22) esta mesa se convierte en el **plan de pruebas** de tu plataforma: ¿qué incidente de la mesa debe poder diagnosticar tu arquitectura en menos de 15 minutos?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa y ejecutable (redefine las funciones y vuelve a lanzar la rúbrica).

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
def find_window(x, min_len=5):
    x = np.asarray(x, float); n = len(x); sd = x.std() or 1.0
    c = np.concatenate([[0], np.cumsum(x)]); best, win = 0.0, (0, n)
    for a in range(0, n - min_len + 1):
        for b in range(a + min_len, n + 1):
            n_in, n_out = b - a, n - (b - a)
            if n_out < min_len: continue
            m_in, m_out = (c[b] - c[a]) / n_in, (c[n] - (c[b] - c[a])) / n_out
            t = (m_in - m_out) / (sd * np.sqrt(1 / n_in + 1 / n_out))
            if abs(t) > abs(best): best, win = t, (a, b)
    if win[0] == 0 and win[1] < n: win = (win[1], n)
    return win[0], win[1], float(best)

def bubble_up(df, bad, cols, min_n=5):
    out = []
    for col in cols:
        pb = df[bad][col].value_counts(normalize=True); pg = df[~bad][col].value_counts(normalize=True); nb = df[bad][col].value_counts()
        for v, share in pb.items():
            if nb[v] >= min_n: out.append({"atributo": col, "valor": v, "cuota en malos": round(share, 3), "cuota en buenos": round(pg.get(v, 0.0), 3), "lift": round(share / (pg.get(v, 0.0) + 0.01), 1), "n malos": int(nb[v])})
    return pd.DataFrame(out).sort_values("lift", ascending=False).reset_index(drop=True)

def diagnose_a(b):
    M = pd.DataFrame(b.metrics); a, e, _ = find_window(M["p95_s"])
    sf = span_frame(b); err = sf[(sf["status"] == "ERROR") & sf["name"].str.startswith("execute_tool")]
    culprit = err.groupby("service").size().idxmax()                                      # el servicio con más herramientas en error
    agent = culprit.replace("supportops-", "")
    tt = {r["trace_id"]: r["start_min"] for r in I.trace_table(b.spans, b.t0)}
    c = collections.Counter(s["trace_id"] for s in b.spans if s["name"] == f"invoke_agent {agent}" and s["kind"] == "CLIENT")
    inw = [v for t, v in c.items() if a <= tt[t] < e]; out = [v for t, v in c.items() if tt[t] < a]
    return {"cause": "dependency_timeout_retry_storm", "start_min": a, "end_min": e, "culprit": culprit, "evidence": {"retry_amplification": np.mean(inw) / np.mean(out)}, "mitigation": "circuit_breaker_fallback"}

def diagnose_b(b):
    M = pd.DataFrame(b.metrics); a, e, _ = find_window(M["cost_per_ticket"])
    tt = pd.DataFrame(I.trace_table(b.spans, b.t0))
    loops = tt[(tt["n_tools"] >= 5) & tt["tools"].apply(lambda t: len(set(t)) == 1)]
    tool = collections.Counter(t[0] for t in loops["tools"]).most_common(1)[0][0]
    tk = pd.DataFrame(b.extra["tickets"]); tk["m"] = (tk["created_at"] - b.t0) / 60
    ratio = tk[(tk["m"] >= a) & (tk["m"] < e)]["cost_usd"].mean() / tk[tk["m"] < a]["cost_usd"].mean()
    return {"cause": "tool_loop", "start_min": a, "end_min": e, "culprit": tool, "evidence": {"loop_traces": len(loops), "cost_ratio": ratio}, "mitigation": "loop_guard"}

def diagnose_c(b):
    sf = span_frame(b); ia = sf[sf["name"].str.startswith("invoke_agent")].copy()
    tools = sf[sf["name"].str.startswith("execute_tool")].groupby("trace_id").size()
    ia["n_tools"] = ia["trace_id"].map(tools).fillna(0); ia = ia[ia["intent"].isin(["order_status", "refund"])]
    new = sorted(v for v in ia["version"].dropna().unique())[-1]                       # la versión «más nueva» presente (v2)
    start = int(ia[ia["version"] == new]["minute"].min())
    full = ia[ia["version"] == new].groupby("minute").size().index.min()
    share_by_v = ia.groupby("version")["n_tools"].apply(lambda s: (s == 0).mean())
    late = ia[ia["minute"] >= max(30, start)]                                             # cuando la versión nueva lleva todo el tráfico
    after = (late["n_tools"] == 0).mean()
    return {"cause": "silent_prompt_regression", "start_min": start, "end_min": 59, "culprit": new, "evidence": {"no_tool_share_after": after}, "mitigation": "rollback_release"}

def diagnose_d(b):
    M = pd.DataFrame(b.metrics); a, e, _ = find_window(M["active_series"])
    lv = b.extra["label_values"]; early, late = sorted(lv)
    cand = {(m, l): n for m in lv[late] for l, n in lv[late][m].items() if l not in lv[early].get(m, {})}
    culprit = max(cand.items(), key=lambda kv: kv[1])[0][1]
    growth = M["active_series"].iloc[-5:].mean() / M["active_series"].iloc[a - 10:a - 1].mean()
    return {"cause": "metric_label_cardinality", "start_min": a, "end_min": e, "culprit": culprit, "evidence": {"series_growth": growth}, "mitigation": "drop_label_collector"}

def diagnose_e(b):
    M = pd.DataFrame(b.metrics); a, e, _ = find_window(M["traces_per_ticket"])
    tt = pd.DataFrame(I.trace_table(b.spans, b.t0)); after = tt[tt["start_min"] >= a + 2]
    share = (~after["root"].str.startswith("invoke_workflow")).mean()
    return {"cause": "context_propagation_broken", "start_min": a, "end_min": e, "culprit": None, "evidence": {"orphan_share_after": share}, "mitigation": "restore_propagation"}

D = {c: run(c, f) for c, f in [("INC-A", diagnose_a), ("INC-B", diagnose_b), ("INC-C", diagnose_c), ("INC-D", diagnose_d), ("INC-E", diagnose_e)]}
display(run_rubric())